# Marco II — Aprendizado de Máquina aplicado à Malha Aérea

**Projeto de Inteligência Artificial — Fase II**

**Integrantes**
- Sofia de Carvalho Brito
- Pedro de Faria Mello
- Victor Sousa Pereira

## Objetivo

Nesta fase, o agente deixa de considerar somente uma métrica fixa para escolher uma rota. Um modelo de aprendizado de máquina é treinado para estimar o tempo de percurso de cada voo, e o algoritmo A* utiliza essas previsões como custos das arestas.

O fluxo do experimento é:

1. Carregar a malha aérea e gerar um conjunto de dados sintético.
2. Preparar os dados e comparar modelos de regressão.
3. Avaliar os modelos com validação cruzada e métricas de erro.
4. Construir um mapa de custos com os tempos previstos.
5. Executar o A* usando `predicted_time` e comparar as rotas.

> **Importante:** os tempos deste experimento são sintéticos. Eles são gerados por uma simulação definida no projeto e não representam registros reais de companhias aéreas.

## 1. Ambiente de execução

Este notebook foi preparado para execução no Google Colab em uma sessão nova. A primeira etapa clona o repositório do grupo, instala as dependências e define a pasta do projeto.

Execute as células em ordem, usando **Ambiente de execução → Executar tudo**.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/soficb14/projeto-ia-malhas-aereas.git"
REPO_DIR = Path("/content/projeto-ia-malhas-aereas")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", REPO_URL, str(REPO_DIR)],
    check=True
)

os.chdir(REPO_DIR)
print("Diretório do projeto:", Path.cwd())
print("Arquivos principais:")
for path in sorted(Path("src").rglob("*.py")):
    print("-", path)

In [ ]:
%pip install -q -r requirements.txt

## 2. Verificação da implementação

Antes de executar os experimentos, verificamos se os testes automatizados do repositório continuam passando. Isso ajuda a detectar problemas na implementação antes de interpretar os resultados do aprendizado de máquina.

In [ ]:
!python3 -m pytest -v

## 3. Carregamento da malha aérea

Utilizamos o mapa médio da malha aérea. Cada aeroporto é um estado do problema e cada voo representa uma conexão possível entre dois estados.

O dataset sintético será gerado a partir dos voos existentes nessa malha.

In [ ]:
from src.environment.loader import load_network

AIRPORTS_PATH = "data/maps/airports.json"
MAP_PATH = "data/maps/mapa_medio.json"

network = load_network(MAP_PATH, AIRPORTS_PATH)

num_airports = len(network.airports)
num_flights = sum(len(flights) for flights in network.flights.values())

print(f"Aeroportos: {num_airports}")
print(f"Voos direcionados: {num_flights}")
print("Aeroportos:", ", ".join(sorted(network.airports.keys())))

## 4. Geração do conjunto de dados

O gerador cria várias observações sintéticas para cada voo. Além da distância, cada observação contém características que podem influenciar o tempo:

- tipo de aeronave;
- índice de congestionamento da rota;
- condição climática;
- componente do vento;
- período do dia.

A variável que queremos prever é `real_elapsed_time`, medida em minutos. O gerador também introduz alguns valores ausentes em variáveis de entrada para que o pré-processamento possa tratá-los.

In [ ]:
from src.environment.dataset import generate_synthetic_flight_dataset

dataset = generate_synthetic_flight_dataset(
    network,
    samples_per_flight=20,
    seed=42,
    missing_rate=0.03
)

print("Dimensões do dataset:", dataset.shape)
display(dataset.head())

### 4.1. Conhecendo os dados

Nesta etapa observamos os tipos de variáveis, a distribuição do tempo de voo e a quantidade de valores ausentes antes do pré-processamento.

In [ ]:
display(dataset.info())
display(dataset.describe(include="all").T)

missing = dataset.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("valores_ausentes"))

print(
    "Tempo de voo (minutos):",
    f"mínimo={dataset['real_elapsed_time'].min():.2f},",
    f"média={dataset['real_elapsed_time'].mean():.2f},",
    f"máximo={dataset['real_elapsed_time'].max():.2f}"
)

### 4.2. Variáveis utilizadas

O modelo utiliza as variáveis abaixo como entrada (**X**) e aprende a estimar o tempo real simulado (**y**).

| Papel | Variáveis |
|---|---|
| Numéricas | `distance`, `route_congestion_index`, `wind_component` |
| Categóricas | `aircraft_type`, `weather_condition`, `time_of_day` |
| Alvo | `real_elapsed_time` |

As colunas `origin` e `destination` são preservadas para identificar os grupos de rotas na validação cruzada e para relacionar as previsões aos voos da malha.

## 5. Pré-processamento e treinamento

O pipeline de pré-processamento evita que informações do conjunto de teste sejam utilizadas durante o ajuste dos transformadores.

- Nas variáveis numéricas, valores ausentes são substituídos pela mediana e os valores são padronizados.
- Nas variáveis categóricas, valores ausentes são substituídos pela categoria mais frequente e as categorias são convertidas com One-Hot Encoding.

Comparamos um baseline de média, Regressão Linear e Random Forest. A escolha do melhor modelo entre os dois modelos de aprendizado é feita pelo menor MAE médio da validação cruzada.

In [ ]:
from src.environment.model import train_and_evaluate, error_analysis

best_pipeline, model_results, test_data = train_and_evaluate(
    dataset,
    seed=42,
    n_splits=5,
    group_by_route=True
)

X_test, y_test, y_pred = test_data

display(model_results.sort_values("cv_mae"))
print("Melhor modelo selecionado:", model_results.drop(index="Baseline (média)")["cv_mae"].idxmin())

### 5.1. Como interpretar as métricas

- **MAE:** erro absoluto médio, em minutos. Por exemplo, MAE de 5 significa que as previsões erram, em média, cerca de 5 minutos.
- **RMSE:** raiz do erro quadrático médio. Erros grandes têm maior influência nessa métrica.
- **CV MAE:** MAE médio obtido nas partições da validação cruzada.
- **CV MAE std:** dispersão do MAE entre as partições.

Quanto menores os valores de MAE e RMSE, melhor o desempenho preditivo. O baseline serve como referência simples: os modelos treinados devem ser comparados com ele.

In [ ]:
import matplotlib.pyplot as plt

metricas_plot = model_results[["cv_mae", "cv_rmse", "teste_mae", "teste_rmse"]]
display(metricas_plot)

metricas_plot.plot(kind="bar", figsize=(10, 5))
plt.title("Comparação dos modelos — erros de previsão")
plt.xlabel("Modelo")
plt.ylabel("Erro (minutos)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 6. Análise dos erros

Além de medir o erro geral, analisamos em quais condições o modelo apresenta maior erro absoluto médio. Isso permite observar se determinadas condições — como tempestades, horários de pico ou tipos de aeronave — são mais difíceis de prever.

In [ ]:
error_groups = error_analysis(X_test, y_test, y_pred)

for variable, values in error_groups.items():
    print(f"Erro absoluto médio por {variable}:")
    display(values.to_frame("erro_absoluto_medio_min"))

## 7. Verificação da heurística do A*

O A* utiliza:

\[
f(n) = g(n) + h(n)
\]

- \(g(n)\) é o custo acumulado desde a origem;
- \(h(n)\) é a estimativa do custo restante até o destino.

Para a métrica `predicted_time`, a heurística estima o tempo mínimo em linha reta, considerando a velocidade máxima de 900 km/h. O mapa de custos também aplica esse limite mínimo às previsões de cada voo.

A função `check_heuristic` verifica se o custo de cada aresta é pelo menos igual ao limite inferior calculado entre os aeroportos conectados.

In [ ]:
from src.search.heuristics import check_heuristic

for metric in ["distance", "estimated_time"]:
    violations = check_heuristic(network, metric=metric)
    print(f"{metric}: {len(violations)} violações")

## 8. Integração do modelo ao A*

Agora usamos o modelo treinado para estimar o tempo de cada voo da malha. O contexto abaixo representa condições favoráveis: aeronave narrow-body, baixo congestionamento, céu claro, vento neutro e período fora do pico.

O resultado de `build_cost_map` é um dicionário que associa cada conexão `(origem, destino)` ao tempo previsto em minutos.

In [ ]:
from src.environment.model import build_cost_map
from src.search.astar import a_star_search
from src.search.heuristics import check_heuristic

contexto_bom = {
    "aircraft_type": "Narrow-body",
    "route_congestion_index": 1,
    "weather_condition": "Clear",
    "wind_component": 0,
    "time_of_day": "Off-peak"
}

cost_map_good = build_cost_map(
    network,
    best_pipeline,
    context=contexto_bom
)

network.set_learned_costs(cost_map_good)

print("Custos previstos:", len(cost_map_good))
print("Exemplos:")
for key, value in list(cost_map_good.items())[:5]:
    print(f"{key[0]} -> {key[1]}: {value:.2f} min")

violations_predicted = check_heuristic(network, metric="predicted_time")
print("Violações da heurística predicted_time:", len(violations_predicted))

## 9. Comparação entre rota por distância e rota por tempo previsto

Executamos o A* duas vezes com a mesma origem e destino:

1. `distance`: minimiza a distância total percorrida.
2. `predicted_time`: minimiza o tempo total previsto pelo modelo.

Assim, conseguimos observar se considerar as condições dos voos altera a decisão do agente.

In [ ]:
# Escolhemos aeroportos existentes no mapa e usamos o mesmo par nas duas buscas.
START = "DFW"
GOAL = "LAX"

if START not in network.airports or GOAL not in network.airports:
    raise ValueError("A origem ou o destino escolhido não está presente no mapa médio.")

distance_result = a_star_search(
    network,
    start=START,
    goal=GOAL,
    metric="distance"
)

predicted_result_good = a_star_search(
    network,
    start=START,
    goal=GOAL,
    metric="predicted_time"
)

def route_codes(result):
    return " -> ".join(airport.code for airport in result.path)

comparison_good = {
    "Métrica": ["Distância", "Tempo previsto"],
    "Rota": [route_codes(distance_result), route_codes(predicted_result_good)],
    "Custo": [distance_result.cost, predicted_result_good.cost],
    "Nós expandidos": [
        getattr(distance_result, "expanded_nodes", None),
        getattr(predicted_result_good, "expanded_nodes", None)
    ]
}

display(__import__("pandas").DataFrame(comparison_good))
print("A rota mudou?", route_codes(distance_result) != route_codes(predicted_result_good))

## 10. Experimento com condições desfavoráveis

Agora alteramos o contexto para representar condições desfavoráveis: congestionamento elevado, tempestade, vento contrário e horário de pico.

O modelo recalcula os custos previstos e o A* executa novamente a busca. Essa comparação mostra como a estimativa de tempo pode influenciar a escolha da rota.

In [ ]:
contexto_ruim = {
    "aircraft_type": "Narrow-body",
    "route_congestion_index": 5,
    "weather_condition": "Storm",
    "wind_component": -20,
    "time_of_day": "Peak"
}

cost_map_bad = build_cost_map(
    network,
    best_pipeline,
    context=contexto_ruim
)

network.set_learned_costs(cost_map_bad)

predicted_result_bad = a_star_search(
    network,
    start=START,
    goal=GOAL,
    metric="predicted_time"
)

comparison_contexts = __import__("pandas").DataFrame([
    {
        "Contexto": "Favorável",
        "Rota": route_codes(predicted_result_good),
        "Tempo previsto (min)": predicted_result_good.cost
    },
    {
        "Contexto": "Desfavorável",
        "Rota": route_codes(predicted_result_bad),
        "Tempo previsto (min)": predicted_result_bad.cost
    }
])

display(comparison_contexts)
print(
    "A rota mudou entre os contextos?",
    route_codes(predicted_result_good) != route_codes(predicted_result_bad)
)

## 11. Evento pontual em um trecho

Além de alterar o contexto geral, o agente pode receber um evento localizado. Neste exemplo, aplicamos uma tempestade a um único trecho da rota encontrada pela busca por distância.

O custo desse trecho é recalculado com as informações específicas do evento, sem alterar o contexto dos demais voos.

In [ ]:
distance_route_codes = [airport.code for airport in distance_result.path]

if len(distance_route_codes) >= 2:
    affected_edge = (distance_route_codes[0], distance_route_codes[1])

    cost_map_storm = build_cost_map(
        network,
        best_pipeline,
        context=contexto_bom,
        overrides={
            affected_edge: {
                "weather_condition": "Storm",
                "route_congestion_index": 5
            }
        }
    )

    network.set_learned_costs(cost_map_storm)
    storm_result = a_star_search(
        network,
        start=START,
        goal=GOAL,
        metric="predicted_time"
    )

    print("Trecho afetado:", f"{affected_edge[0]} -> {affected_edge[1]}")
    print("Rota após evento:", route_codes(storm_result))
    print("Tempo previsto:", round(storm_result.cost, 2), "min")
else:
    print("A rota encontrada não possui trecho para aplicar o evento.")

## 12. Busca por pares de aeroportos em que a rota muda

Uma única origem e destino podem não demonstrar uma mudança de rota. Por isso, verificamos vários pares de aeroportos do mapa médio.

Para cada par, comparamos a rota por distância com a rota por tempo previsto no contexto favorável. Registramos os pares em que as sequências de aeroportos são diferentes.

In [ ]:
network.set_learned_costs(cost_map_good)

route_changes = []
airport_codes = sorted(network.airports.keys())

for origin in airport_codes:
    for destination in airport_codes:
        if origin == destination:
            continue

        try:
            by_distance = a_star_search(
                network, start=origin, goal=destination, metric="distance"
            )
            by_time = a_star_search(
                network, start=origin, goal=destination, metric="predicted_time"
            )
        except (ValueError, KeyError):
            continue

        route_distance = route_codes(by_distance)
        route_time = route_codes(by_time)

        if route_distance != route_time:
            route_changes.append({
                "Origem": origin,
                "Destino": destination,
                "Rota por distância": route_distance,
                "Rota por tempo previsto": route_time,
                "Custo por distância": by_distance.cost,
                "Tempo previsto (min)": by_time.cost
            })

route_changes_df = __import__("pandas").DataFrame(route_changes)
print("Pares com mudança de rota:", len(route_changes_df))
display(route_changes_df.head(20))

## 13. Salvamento dos resultados

Salvamos os principais resultados em CSV para facilitar a reprodução, a elaboração do relatório e a conferência dos números apresentados.

In [ ]:
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

dataset.to_csv(RESULTS_DIR / "fase2_dataset_notebook.csv", index=False)
model_results.to_csv(RESULTS_DIR / "fase2_metricas_notebook.csv")
route_changes_df.to_csv(RESULTS_DIR / "fase2_rotas_que_mudam_notebook.csv", index=False)

print("Arquivos salvos em:", RESULTS_DIR.resolve())
print("Arquivos:")
for path in sorted(RESULTS_DIR.glob("*notebook.csv")):
    print("-", path.name)

## 14. Análise dos resultados

Após executar todas as células, preencham esta análise com os valores efetivamente exibidos pelo notebook.

1. **Comparação dos modelos:** qual modelo apresentou o menor CV MAE? Ele superou o baseline? Informe os valores de MAE e RMSE.
2. **Análise dos erros:** em quais condições o modelo apresentou maior erro absoluto médio? O que pode explicar essa dificuldade?
3. **Integração com A\*:** a rota por tempo previsto foi diferente da rota por distância? E a mudança do contexto favorável para o desfavorável alterou a rota ou apenas o custo?
4. **Limitações:** os dados são sintéticos e dependem das regras usadas pelo gerador. Portanto, os resultados demonstram o funcionamento do método dentro da simulação, mas não comprovam desempenho em operações aéreas reais.

### Limitações metodológicas a registrar

- Os tempos são simulados por uma fórmula definida no projeto, e não coletados de voos reais.
- A validação cruzada pode agrupar observações por rota (`GroupKFold`), mas a separação final treino/teste implementada no módulo ainda é aleatória por amostra. Assim, uma mesma rota pode aparecer nos dois conjuntos.
- O resultado depende das variáveis e dos parâmetros escolhidos para a simulação.
- A heurística temporal usa a velocidade máxima de 900 km/h como limite inferior. A verificação automatizada deve ser executada e seu resultado registrado.

## 15. Conclusão

Na Fase II, o agente passa a utilizar aprendizado de máquina para estimar o tempo de cada voo. Os modelos são treinados com características operacionais simuladas e comparados por métricas de regressão. Em seguida, as previsões são incorporadas aos custos das arestas da malha aérea, permitindo que o A* procure uma rota de menor tempo previsto, em vez de considerar somente a distância.

A conclusão definitiva deve ser escrita após a execução do notebook, utilizando os resultados observados e reconhecendo as limitações do dataset sintético.